# EXP-2: YOLOv5n (Anchor-based) — TVDS
**Người thực hiện:** Phúc (Backend & Data)  
**Giao thức chung E1:** Dataset gốc (368 train / 65 val, 7 classes), `imgsz=640`, `seed=0`, `batch=16`.  
**Lưu ý:** Dùng repo chính thức `ultralytics/yolov5` (anchor-based) để đối chiếu đúng với `YOLOv8` (anchor-free).

In [ ]:
# 1. Kiểm tra GPU T4
!nvidia-smi

In [ ]:
# 2. Mount Google Drive & giải nén dataset gốc
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/Traffic_Project/all_datasets.zip"
EXTRACT_DIR = "/content/dataset"

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"Không thấy {ZIP_PATH}. Nếu thư mục Traffic_Project đang ở 'Được chia sẻ với tôi' (Shared with me), "
        "hãy chuột phải vào Traffic_Project -> Tổ chức -> Thêm lối tắt vào Drive (MyDrive)."
    )

!mkdir -p "{EXTRACT_DIR}"
!unzip -q -o "{ZIP_PATH}" -d "{EXTRACT_DIR}"
print("Đã giải nén xong vào:", EXTRACT_DIR)

In [ ]:
# 3. Chuẩn hóa thư mục valid và tạo /content/data_helmet.yaml (7 classes)
import os, shutil

base = "/content/dataset/content/smart-helmet-detection-using-dl/data"
if os.path.exists(f"{base}/vaid") and not os.path.exists(f"{base}/valid"):
    shutil.move(f"{base}/vaid", f"{base}/valid")

yaml_text = f"""train: {base}/train
val: {base}/valid
nc: 7
names:
- driver_with_helmet
- bike
- driver
- passenger_with_helemt
- passenger
- driver_without_helmet
- passenger_without_helemt
"""
with open("/content/data_helmet.yaml", "w") as f:
    f.write(yaml_text)

print("train exists:", os.path.exists(f"{base}/train"))
print("val exists:", os.path.exists(f"{base}/valid"))

In [ ]:
# 4. Clone repo ultralytics/yolov5 (anchor-based) và cài thư viện
!git clone -q https://github.com/ultralytics/yolov5 /content/yolov5
!pip install -q -r /content/yolov5/requirements.txt

In [ ]:
# 5. SMOKE TEST (3 epochs) — chạy xong báo SMOKE_MINUTES cho Khoa ngay!
import time, subprocess

t0 = time.time()
cmd = [
    "python", "/content/yolov5/train.py",
    "--img", "640",
    "--batch", "16",
    "--epochs", "3",
    "--data", "/content/data_helmet.yaml",
    "--weights", "yolov5n.pt",
    "--seed", "0",
    "--project", "/content/runs/e1",
    "--name", "yolov5n_smoke",
    "--exist-ok"
]
subprocess.run(cmd, check=True)
print("SMOKE_MINUTES:", round((time.time() - t0) / 60, 2))

In [ ]:
# 6. TRAIN THẬT EXP-2 — Đổi EPOCHS thành 30 hoặc 50 theo chốt của Khoa sau smoke test
EPOCHS = 50  # <-- Sửa thành 30 nếu Khoa chốt 30 epochs

import time, subprocess
t0 = time.time()
cmd = [
    "python", "/content/yolov5/train.py",
    "--img", "640",
    "--batch", "16",
    "--epochs", str(EPOCHS),
    "--data", "/content/data_helmet.yaml",
    "--weights", "yolov5n.pt",
    "--seed", "0",
    "--project", "/content/runs/e1",
    "--name", "yolov5n_e1",
    "--exist-ok"
]
subprocess.run(cmd, check=True)
print("TRAIN_MINUTES:", round((time.time() - t0) / 60, 2))

In [ ]:
# 7. Đánh giá lại từ best.pt để lấy bảng P/R/mAP50/mAP50-95 tổng và từng class (nhất là class 3, 5, 6)
!python /content/yolov5/val.py --weights /content/runs/e1/yolov5n_e1/weights/best.pt --data /content/data_helmet.yaml --img 640 --task val

In [ ]:
# 8. Đóng gói toàn bộ artifact EXP-2 (best.pt, results.csv, opt.yaml, confusion_matrix.png...) và tải về
!zip -rq /content/exp2_yolov5n_e1.zip /content/runs/e1/yolov5n_e1
!cp /content/exp2_yolov5n_e1.zip /content/drive/MyDrive/Traffic_Project/exp2_yolov5n_e1.zip

from google.colab import files
files.download("/content/exp2_yolov5n_e1.zip")